In [ ]:
# ============================================
# PACKAGE INSTALLATIONS
# ============================================

import os # to talk to os, particularly to use env. vars
import certifi # gives trusted list of SSL certificates -> id for the website, tells that it can be trusted, lets Python to use this certificate bundle in HTTPS requests, network security setting
import requests # https calls
from dotenv import load_dotenv  # read env. variables from .env and load them into Python

from langchain_google_genai import ChatGoogleGenerativeAI # langchain wrapper around OpenAI models, langchain connection to OpenAI chat model
from langchain_tavily import TavilySearch
from langchain.tools import tool # to refine a finished program by adding a custom tool of weather info search
import streamlit as st

In [ ]:
from langchain.agents import create_agent # we need to import some agents functionality, react -> agent has an ability to reason and perform actions

In [ ]:
# ==========================
# LOAD ENVIRONMENT VARIABLES
# ==========================
os.environ["SSL_CERT_FILE"] = certifi.where() # uploads certified updates path instead of old one, may happen in Windows -> path issue
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY") # retrives the key, we do not put the secret directly into our source code
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [ ]:
# =======================================
# SEARCH TOOL
# ======================================= better for agent to have tools
search_tool = TavilySearch(max_results=2)


In [ ]:
# ============================
# WEATHER INFORMATION
# ============================
# custom tool, "take this normal Python function and expose it to the agent as a tool, without the decorator, it's just a Python function, but now LangChain knows the agent/LLM can call this function"
@tool
def get_weather_data(city: str) -> str:
    """Fetch the current weather information of the city""" # docstring in function as a tool helps agent to understand what the tool does, it matters a lot in agentic systems
    url= (
        f"http://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )
    response = requests.get(url) # it will give us a structured JSON file
    data = response.json() # Python converts to a dictionary-like object
    if "current" not in data:
        return f"Could not fetch weather data for {city}"
    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']} degrees Celsius\n"
    )
        
    

In [ ]:
search_tool.invoke("What is the weather in Tokyo in October") # testing the tool, invoke = вызывать, вызов

[{'url': 'https://www.weather25.com/asia/japan/tokyo?page=month&month=October',
  'content': 'weather25.com\nSearch\nweather in Japan\nRemove from your favorite locations\nAdd to my locations\nShare\nweather in Japan\n\n# Tokyo weather in October 2026\n\nThe temperatures in Tokyo in October are comfortable with low of 15°C and and high up to 22°C.\n\nYou can expect about 3 to 8 days of rain in Tokyo during the month of October. It’s a good idea to bring along your umbrella so that you don’t get caught in poor weather.\n\nOur weather forecast can give you a great sense of what weather to expect in Tokyo in October 2026.\n\nIf you’re planning to visit Tokyo in the near future, we highly recommend that you review the 14 day weather forecast for Tokyo before you arrive.\n\nTemperatures\nRainy Days\nSnowy Days\nDry Days\nRainfall\n8.9 [...] | Sun | Mon | Tue | Wed | Thu | Fri | Sat |\n ---  ---  --- \n|  |  |  |  | 1 Light rain shower 25° /20° | 2 Sunny 28° /21° | 3 Partly cloudy 26° /21° |

In [ ]:
# =======================================
# LLM
# =======================================
# the brain we use to ask questions

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash", # it's an old of 2022 LLM, so if we don't provide external tools, then it's knowledge is limited, if we provide, 
    # it will give recent info as well as it may search the Internet; but we changed OpenAI API to free Gemini API
    temperature=0.1, # be more deterministic/not random
    api_key=GOOGLE_API_KEY
)

In [ ]:
response = llm.invoke("What is the day today?")
response

In [ ]:
# ==============================
# PROMPT
# ==============================
# prompt is important, it loops until gives final answer -> gives an agent a predefined ReAct-style prompt: loop of T->A->O->T->A->O, etc. -> agent loop
# prompt = hub.pull("hwchase17/react") # recommended/standard tool for this task, but external prompt may be dangerous so we may write manually
prompt = """
    You are a helpful research assistant. 
    Use search for current factual information. 
    Use the weather tool for current weather. 
    Do not invent tool results.
    """


c:\Users\Public\Downloads\anaconda\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [ ]:
# ========================
# TOOLS
# ========================
tools = [search_tool, get_weather_data] # toolbox, the actions LLM is allowed to use

In [ ]:
# ============================
# CREATE AGENT 
# ============================
# agent object
agent = create_agent( 
    model=llm,
    tools=tools,
    system_prompt=prompt, #all instructions
    debug=True # to see the logs of the agent invokation, what it does, what the agent is doing, debug mode, intermediate agent/tool activity
)

In [ ]:
# ================================
# AGENT INVOCATION
# ================================
# modern invokation is message-based, agent maintains whole message memory/state
response = agent.invoke({
    "messages": [
        {
            "role": "user", # who said it? common roles: user, system, assistant, tool
            "content": "Find the capital of Switzerland and its current weather" # what did they say?
        }]})

In [ ]:
print(response["messages"][-1].content) # the final/last message from the state of history of messages from agent execution state